# 5. Does module activity track outcome

In TCGA only. Whatever clears FDR here is what gets frozen and taken to METABRIC; nothing
is re-selected afterwards.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import Config
from src.io_utils import data_path, read_matrix, read_table, results_path

cfg = Config.load("../config.json")

In [ ]:
from src.modules import activity_table, load_modules
from src.survival import (
    design_matrix, fit_module_models, kaplan_meier, logrank_test, median_survival,
    prepare_endpoint, split_by_median,
)

expr = read_matrix(results_path("preprocessed", "tcga_expression_raw.tsv.gz"))
meta = read_table(results_path("preprocessed", "tcga_meta.tsv"), index_col=0)
modules = load_modules(results_path("modules", "modules.tsv"))
{m: len(g) for m, g in modules.items()}

Scored on the unadjusted matrix: a prognostic score should work on the expression a lab
would actually measure. `scripts/05_survival.py --adjusted` repeats it on the adjusted one.

In [ ]:
activity = activity_table(expr, modules, method="rank")
activity.describe().round(3)

In [ ]:
surv = prepare_endpoint(meta, cfg.survival_endpoint, cfg.max_followup_days)
covariates = design_matrix(meta.loc[surv.index], cfg.covariates).dropna()
print(len(surv), "patients,", int(surv["event"].sum()), "events,", covariates.shape[1], "covariates")
covariates.columns.tolist()

In [ ]:
cox = fit_module_models(activity, surv, covariates)
cox[["module", "n", "events", "hr_uni", "ci_lower_uni", "ci_upper_uni", "p_uni", "q_uni"]]

The adjusted column is the one to quote. A module that is only prognostic univariably is
usually a proxy for subtype or stage.

In [ ]:
cox[["module", "hr_uni", "p_uni", "hr_adj", "p_adj_model", "q_adj_model"]]

In [ ]:
module = cox["module"].iloc[0]
score = activity[module].reindex(surv.index).dropna()
arms = split_by_median(score)
frame = surv.loc[score.index]

for arm in ("high", "low"):
    take = frame.loc[arms[arms == arm].index]
    km = kaplan_meier(take["time"], take["event"])
    print(arm, "n =", len(take), "median =", median_survival(km))
print("log-rank p =", logrank_test(frame["time"], frame["event"], arms)["p_value"])